# 🦋 Verifying the findings with LAM dataset

In [5]:
import time
import json
import os
import re
import pandas as pd
import base64
from IPython.display import display, Image
from PIL import Image as PILImage

In [8]:
path = os.path.dirname(os.getcwd())
path

'/Users/serenekim/Desktop/PhD/img-analysis_seorin_project'

In [ ]:
# GT
with open(f'{path}/data/LAM/lines/transcriptions.json', 'r') as f:
    tr = json.load(f)

transcriptions = pd.DataFrame(tr)

# CER/ BLEU

In [12]:
transcriptions

,decade_id,height,img,nameset,text,width
0,0,130,002_02_00.jpg,train,Lettera di Lod. Antonio,1120
1,0,147,002_02_01.jpg,train,di Giacomo Bianchi,986
2,1,112,002_04_00.jpg,train,Arona 30. 9bre. 94.,657
3,1,82,002_04_01.jpg,train,Sorella amatissima,544
4,1,93,002_04_02.jpg,train,Veramente il vostro caso è degno di compassion...,1654
...,...,...,...,...,...,...
25818,3,41,094_4405_07.jpg,test,Di VS. # e Revma,345
25819,3,46,094_4405_08.jpg,test,Mod.a 2. Marzo. 1717.,336
25820,3,41,094_4405_09.jpg,test,A Mons.r Battelli Arciv.o,379
25821,3,41,094_4405_10.jpg,test,d'Amasia e Segr.o de',285


In [15]:
epoch = []
epoch_num = 13
for file in os.listdir(f'{path}/results/epoch{epoch_num}'):
    if file.endswith('.txt'):
        name = file.split(f'_epoch_{epoch_num}_')[0]
        
        with open(f'{path}/results/epoch{epoch_num}/{file}', 'r') as f:
            text = f.read()
        epoch.append({'img': name, 'text': text})
epoch_df = pd.DataFrame(epoch)

In [17]:
epoch_df

,img,text
0,010_351_11.jpg,Mod.a 22. Xbre 1727.
1,021_894_31.jpg,Monsieur.
2,021_892_31.jpg,"una'è ultimamente. sala d'aver'aveso,"
3,021_896_00.jpg,un 22. Illa Carità Cristiana.
4,006_297_12.jpg,mente l'avviso a V.A.S. c. che. cio al sovano
...,...,...
697,030_1137_06.jpg,comcommandcomptimComcomecconcocongcontincomedd...
698,005_279_01.jpg,tro con lei per questo nuovo merito
699,023_918_13.jpg,più volto da comparire in pubblico Prego
700,021_894_05.jpg,la libertà di screditare le # divività Pape-


In [18]:
from evaluate import load

cer_metric =load("cer")
bleu_metric = load("bleu")  

In [40]:
import re
import unidecode

sub = 'img'

bleu_perline = pd.DataFrame()
cer_perline = pd.DataFrame()


pred = epoch_df
df_filtered = transcriptions[transcriptions[sub].isin(pred[sub].unique())] 

bleu_scores = []  
cer_scores = [] 

for id in df_filtered[sub].unique():
    # Extract the text as a single string, not as an array
    pred_text = pred[pred[sub] == id]['text'].values
    ref_text = df_filtered[df_filtered[sub] == id]['text'].values

    # Ensure the predictions and references are passed as a list of strings
    if len(pred_text) > 0 and len(ref_text) > 0:  # Check if both texts are not empty
        pred_text = pred_text[0]
        ref_text = ref_text[0]

        # Check for NaN values 
        if pd.notna(pred_text) and pd.notna(ref_text):
            # Remove multiple consecutive hyphens (e.g., "----" or "- - - - -")
            pred_text = re.sub(r'[-\s]+', ' ', pred_text)
            ref_text = re.sub(r'[-\s]+', ' ', ref_text)

                # Remove single hyphens between words (e.g., "vingt-et-un" -> "vingt et un")
            pred_text = re.sub(r'\b(\w+)-(\w+)\b', r'\1 \2', pred_text)
            ref_text = re.sub(r'\b(\w+)-(\w+)\b', r'\1 \2', ref_text)

            # Strip white spaces
            pred_text = pred_text.strip()
            ref_text = ref_text.strip()
            
            pred_text = unidecode.unidecode(pred_text).lower()
            ref_text = unidecode.unidecode(ref_text).lower()

            # Ensure texts are not empty after stripping
            if pred_text and ref_text:
                bleu_metrics = bleu_metric.compute(predictions=[pred_text], references=[ref_text], max_order=2)
                cer_metrics = cer_metric.compute(predictions=[pred_text], references=[ref_text])
            else:
                bleu_metrics = {'bleu': 0.0}  # Assign a default value if texts are empty
                cer_metrics = 1.0
        else:
            bleu_metrics = {'bleu': 0.0}  # Assign a default value if texts are NaN
            cer_metrics = 1.0
    else:
        bleu_metrics = {'bleu': 0.0}  # Assign a default value if texts are empty
        cer_metrics = 1.0

    bleu_scores.append({
            'id': id,
            **bleu_metrics
        })
    cer_scores.append({
            'id': id,
            'cer': cer_metrics
        })

bleu_perline = pd.concat([bleu_perline, pd.DataFrame(bleu_scores)], ignore_index=True)
cer_perline = pd.concat([cer_perline, pd.DataFrame(cer_scores)], ignore_index=True)


In [39]:
bleu_perline.to_csv(f'{path}/results/bleu_perline_LAM_TrOCR_n1_epoch{epoch_num}_output2.csv', index=False)
# cer_perline.to_csv(f'{path}/results/cer_perline_LAM_TrOCR_epoch{epoch_num}_output2.csv', index=False)

In [41]:
bleu_perline['bleu'].agg(['mean', 'std', 'var'])

mean    0.006687
std     0.044570
var     0.001986
Name: bleu, dtype: float64

In [38]:
cer_perline['cer'].agg(['mean', 'std', 'var'])

mean    2.408281
std     3.123589
var     9.756808
Name: cer, dtype: float64